# MammoDiffusion LDM v3 - architettura stile SD + v-prediction (+ min-SNR)

Terza iterazione del diffusore *from scratch*, pensata per essere un esperimento
**completo e affidabile** allo stesso livello di `07_LDM_SDVAE_Extra1361`: training,
generazione, filtro e valutazione usano tutti esplicitamente la stessa parameterization,
cosi' le metriche finali sono coerenti con come il modello e' stato addestrato.

Rispetto a `07_LDM_SDVAE_Extra1361` (latenti prodotti dal VAE preaddestrato di Stable
Diffusion 2.1 + U-Net Keras riaddestrata) cambiano tre cose:

1. **Architettura U-Net**: [`ldm_v3_unet_keras.py`](ldm_v3_unet_keras.py) sostituisce ogni `Conv2DTranspose` con `UpSampling2D(nearest) + Conv2D(3x3)`, adotta un `ResBlock` in stile SD (`GroupNorm -> SiLU -> Conv3x3`, embedding iniettato via FiLM) e usa `SiLU` come attivazione ovunque. Il downsampling resta `Conv2D stride=2` (gia' presente in v2).
2. **v-prediction** (Salimans & Ho 2022): la rete predice `v = sqrt(ab)*eps - sqrt(1-ab)*x0` invece del solo rumore. E' la parameterization usata da SD 2.1 e riduce gli artefatti a guidance scale elevata.
3. **Min-SNR-gamma weighting** (Hang et al. 2023): la loss simple viene pesata per campione con `w(t) = min(SNR(t), gamma) / SNR(t)` per eps o `min(SNR(t), gamma) / (SNR(t)+1)` per v, riducendo l'enfasi sui timestep a rumore basso (dove la loss e' facile) e accelerando la convergenza. Default `gamma=5`.

**Importante — coerenza eps/v-prediction:** `train_ldm_v2.py` addestra con `--parameterization v`,
quindi *tutte* le fasi successive (`evaluate_ldm_v2.py`, `generate_ldm_v2.py`) devono ricevere
esplicitamente `--parameterization v` (e `--unet-version v3`), altrimenti interpreterebbero
l'uscita v-prediction del modello come se fosse rumore puro, generando immagini sbagliate e
metriche non affidabili. Le celle di questo notebook passano questi flag esplicitamente invece
di riusare alla cieca le celle di `07_LDM_SDVAE_Extra1361` (che assumono `eps`): vedi la Sezione 8 in poi.

Il **VAE** puo' essere caricato in due modi:

* `USE_VAE_FT_FROM_03 = False` -> stesso VAE di 07 (SD-VAE congelato, `vae_source="sd_vae_original"`);
* `USE_VAE_FT_FROM_03 = True` -> VAE fine-tuned dal notebook `03_SD21_VAE_FineTuned.ipynb`, risolto automaticamente tra
  piu' path candidati (vedi Sezione 3). Se nessun candidato esiste, il notebook si ferma con un
  errore chiaro invece di ricadere silenziosamente sul VAE originale.

**Cartelle:** `experiments/diffusers/08_ldm_v3_sdvae_fromscratch/`, `results/diffusers/08_ldm_v3_sdvae_fromscratch/`,
immagini finali in `data/synthetic/fromscratch_v3/{positive, negative}/`.


## 1. Selezione GPU

In [1]:
# CUDA + XLA setup: esegui questa cella prima di qualsiasi import TensorFlow.
# Non seleziona alcuna GPU: CUDA_VISIBLE_DEVICES, se necessario, va definita esternamente.
import os
import sys
from pathlib import Path as _Path

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"

_libdevice_candidates = [
    _Path(sys.prefix) / "nvvm" / "libdevice" / "libdevice.10.bc",
    _Path(os.environ["MAMMODIFFUSION_CUDA_ROOT"]) / "nvvm" / "libdevice" / "libdevice.10.bc" if os.environ.get("MAMMODIFFUSION_CUDA_ROOT") else None,
    _Path("/usr/local/cuda/nvvm/libdevice/libdevice.10.bc"),
    _Path("/usr/local/cuda-12.4/nvvm/libdevice/libdevice.10.bc"),
    _Path("/usr/local/cuda-12.2/nvvm/libdevice/libdevice.10.bc"),
    _Path("/usr/lib/nvidia-cuda-toolkit/nvvm/libdevice/libdevice.10.bc"),
]

_cuda_data_dir = None
for _candidate in _libdevice_candidates:
    if _candidate is not None and _candidate.exists():
        _cuda_data_dir = _candidate.parent.parent.parent
        break

if _cuda_data_dir is not None:
    os.environ["XLA_FLAGS"] = f"--xla_gpu_cuda_data_dir={_cuda_data_dir}"
else:
    print("[WARN] libdevice.10.bc non trovato; se TF crasha su JIT, imposta XLA_FLAGS manualmente.")

print("TF_XLA_FLAGS: ", os.environ.get("TF_XLA_FLAGS"))
print("XLA_FLAGS:    ", os.environ.get("XLA_FLAGS", "<non impostato>"))

# GPU dedicata al training; la generazione usa invece GENERATION_GPU_DEVICES nei subprocess.
TRAIN_GPU_VISIBLE_DEVICES = "0"

def training_subprocess_env():
    env = os.environ.copy()
    if TRAIN_GPU_VISIBLE_DEVICES is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(TRAIN_GPU_VISIBLE_DEVICES)
    return env

# Generazione multi-GPU (non influenza il training).
PARALLEL_GENERATION = True
GENERATION_GPU_DEVICES = "auto"
GENERATION_MAX_WORKERS = None

def add_generation_parallel_args(command):
    if PARALLEL_GENERATION:
        command.extend(["--generation-gpus", GENERATION_GPU_DEVICES])
        if GENERATION_MAX_WORKERS is not None:
            command.extend(["--max-generation-workers", str(GENERATION_MAX_WORKERS)])
    else:
        command.extend(["--generation-gpus", "off"])
    return command


TF_XLA_FLAGS:  --tf_xla_auto_jit=0
XLA_FLAGS:     --xla_gpu_cuda_data_dir=/home/fede/miniforge3/envs/tf-gpu


## 2. Setup dipendenze

In [2]:
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'pandas', 'numpy', 'matplotlib', 'scikit-learn', 'pillow', 'gdown',
    'tensorflow', 'scikit-image', 'scipy', 'psutil', 'codecarbon',
    'torch', 'diffusers', 'transformers', 'safetensors', 'accelerate',
    'prdc', 'torch-fidelity',
])

0

## 3. Path progetto ed esperimento (dedicati a v3)

In [3]:
from pathlib import Path
from tempfile import TemporaryDirectory
import shutil
import zipfile
import time
import json

import gdown

PROJECT_NAME = 'MammoDiffusion'
EXPERIMENT_NAME = 'diffusers/08_ldm_v3_sdvae_fromscratch'
RESULTS_STAGE_NAME = 'diffusers/08_ldm_v3_sdvae_fromscratch'
NOTEBOOK_NAME = '08_LDM_v3_SDVAE_FromScratch.ipynb'

SD21_MODEL_DRIVE_ID = '10XRn-bxpp7tP6ROWLYpeCNYJZIaHfMUt'
FORCE_MODEL_REDOWNLOAD = False
PROJECT_ROOT_OVERRIDE = None

# Flag espliciti passati a train_ldm_v2.py / generate_ldm_v2.py / evaluate_ldm_v2.py.
# Definiti qui una sola volta e riusati da tutte le celle sottostanti, cosi' non c'e'
# rischio che training e generazione/valutazione finiscano fuori sincrono.
UNET_VERSION = 'v3'
PARAMETERIZATION = 'v'
USE_MIN_SNR = True
MIN_SNR_GAMMA = 5.0

# Opzione del notebook 08: usa il VAE fine-tuned prodotto dal notebook 03 invece del VAE
# SD standard. Se True, la risoluzione del path e' robusta (piu' candidati) e fallisce
# con un errore esplicito se nessun candidato esiste (nessun fallback silenzioso).
USE_VAE_FT_FROM_03 = False
EXPERIMENT_03_NAME = 'diffusers/03_sd21_vae_finetuned'


def find_project_root(project_name=PROJECT_NAME, override=PROJECT_ROOT_OVERRIDE):
    if override is not None:
        return Path(override).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if candidate.name == project_name or ((candidate / 'data').exists() and (candidate / 'notebooks').exists()):
            return candidate
    for candidate in [cwd / project_name, Path('/content') / project_name, Path.home() / project_name]:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError('Root MammoDiffusion non trovata.')


PROJECT_ROOT = find_project_root()
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
UTILITY_DIR = NOTEBOOKS_DIR / 'utility'
DATA_DIR = PROJECT_ROOT / 'data'
DATA_PROCESSED_DIR = DATA_DIR / 'processed'
EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / EXPERIMENT_NAME
EXPERIMENT_03_DIR = PROJECT_ROOT / 'experiments' / EXPERIMENT_03_NAME
SHARED_PRETRAINED_ROOT = NOTEBOOKS_DIR / 'pretrained_model'
PRETRAINED_MODEL_DIR = SHARED_PRETRAINED_ROOT / 'stable-diffusion-2-1-base'
PRETRAINED_MODEL_ZIP_PATH = SHARED_PRETRAINED_ROOT / 'archives' / 'stable-diffusion-2-1-base.zip'
MODELS_DIR = EXPERIMENT_DIR / 'models'
CHECKPOINTS_DIR = EXPERIMENT_DIR / 'checkpoints_ldm'
LATENTS_DIR = EXPERIMENT_DIR / 'latents'
LOGS_DIR = EXPERIMENT_DIR / 'logs'
RESULTS_DIR = PROJECT_ROOT / 'results' / RESULTS_STAGE_NAME
RESULTS_PLOTS_DIR = RESULTS_DIR / 'plots'
RESULTS_METRICS_DIR = RESULTS_DIR / 'metrics'
RESULTS_ECOTRACKER_DIR = RESULTS_DIR / 'ecotracker'
SYNTHETIC_V3_DIR = DATA_DIR / 'synthetic' / 'fromscratch_v3'
SYNTHETIC_V3_POS_DIR = SYNTHETIC_V3_DIR / 'positive'
SYNTHETIC_V3_NEG_DIR = SYNTHETIC_V3_DIR / 'negative'

for directory in [
    EXPERIMENT_DIR, PRETRAINED_MODEL_ZIP_PATH.parent, MODELS_DIR,
    CHECKPOINTS_DIR, LATENTS_DIR, LOGS_DIR,
    RESULTS_PLOTS_DIR, RESULTS_METRICS_DIR, RESULTS_ECOTRACKER_DIR,
    SYNTHETIC_V3_POS_DIR, SYNTHETIC_V3_NEG_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def vae_ft_03_candidate_dirs():
    """Path candidati (in ordine di preferenza) per il VAE fine-tuned di 03. Non hardcoda
    un solo path: 03 puo' salvare il risultato in posizioni diverse a seconda che sia stato
    rilanciato con resume o meno."""
    return [
        EXPERIMENT_03_DIR / 'vae_finetuning_resume_last' / 'vae_finetuned',
        EXPERIMENT_03_DIR / 'vae_finetuning' / 'vae_finetuned',
        EXPERIMENT_03_DIR / 'pretrained_model_vaeft' / 'stable-diffusion-2-1-base' / 'vae',
    ]


def resolve_vae_ft_03_dir():
    """Restituisce il primo candidato che contiene un config.json (formato diffusers),
    oppure None se nessuno e' valido. Nessun fallback silenzioso: la decisione su cosa fare
    in assenza di un candidato valido spetta al chiamante."""
    for candidate in vae_ft_03_candidate_dirs():
        if (candidate / 'config.json').is_file():
            return candidate
    return None


print('PROJECT_ROOT   :', PROJECT_ROOT)
print('EXPERIMENT_DIR :', EXPERIMENT_DIR)
print('RESULTS_DIR    :', RESULTS_DIR)
print('SYNTHETIC v3   :', SYNTHETIC_V3_DIR)
print('UNET_VERSION   :', UNET_VERSION)
print('PARAMETERIZATION:', PARAMETERIZATION)
print('USE_MIN_SNR    :', USE_MIN_SNR, '| MIN_SNR_GAMMA:', MIN_SNR_GAMMA)
print('USE_VAE_FT_FROM_03:', USE_VAE_FT_FROM_03)

if str(UTILITY_DIR) not in sys.path:
    sys.path.insert(0, str(UTILITY_DIR))
print("CUDA_VISIBLE_DEVICES ereditato:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("GENERATION_GPU_DEVICES richiesto:", GENERATION_GPU_DEVICES)
from parallel_generation_utils import print_gpu_resolution_dry_run
print_gpu_resolution_dry_run(GENERATION_GPU_DEVICES, GENERATION_MAX_WORKERS)


PROJECT_ROOT   : /mnt/MammoDiffusion/MammoDiffusion
EXPERIMENT_DIR : /mnt/MammoDiffusion/MammoDiffusion/experiments/diffusers/08_ldm_v3_sdvae_fromscratch
RESULTS_DIR    : /mnt/MammoDiffusion/MammoDiffusion/results/diffusers/08_ldm_v3_sdvae_fromscratch
SYNTHETIC v3   : /mnt/MammoDiffusion/MammoDiffusion/data/synthetic/fromscratch_v3
UNET_VERSION   : v3
PARAMETERIZATION: v
USE_MIN_SNR    : True | MIN_SNR_GAMMA: 5.0
USE_VAE_FT_FROM_03: False
CUDA_VISIBLE_DEVICES ereditato: None
GENERATION_GPU_DEVICES richiesto: auto
GPU fisiche (nvidia-smi): ['0', '1']
CUDA_VISIBLE_DEVICES ereditato: None
GPU richieste (--generation-gpus): auto
GPU risolte: ['0', '1']
Numero worker: 2


['0', '1']

## 4. Verifica dataset preprocessato (identica a 07)

In [4]:
import pandas as pd

required = [
    DATA_PROCESSED_DIR / 'metadata' / 'train.csv',
    DATA_PROCESSED_DIR / 'metadata' / 'val.csv',
    DATA_PROCESSED_DIR / 'metadata' / 'test.csv',
]
for path in required:
    if not path.exists():
        raise FileNotFoundError(f'Dataset preprocessato mancante: {path}')

for split in ['train', 'val', 'test']:
    df = pd.read_csv(DATA_PROCESSED_DIR / 'metadata' / f'{split}.csv')
    print(split, len(df), df['label'].value_counts().to_dict())

train 2041 {0: 1701, 1: 340}
val 437 {0: 364, 1: 73}
test 438 {0: 365, 1: 73}


## 5. Modello SD2.1 base (per il VAE)

Scarica il modello base se assente. Se `USE_VAE_FT_FROM_03=True`, sostituisce la
sottocartella `vae/` con il VAE fine-tuned di 03 prima di procedere.

In [5]:
MODEL_WEIGHT_ALIASES = {
    'text_encoder': ('model.fp16.safetensors', 'model.safetensors'),
    'unet':         ('diffusion_pytorch_model.fp16.safetensors', 'diffusion_pytorch_model.safetensors'),
    'vae':          ('diffusion_pytorch_model.fp16.safetensors', 'diffusion_pytorch_model.safetensors'),
}


def has_diffusers_structure(d):
    return all((Path(d) / p).exists() for p in ['model_index.json', 'scheduler', 'tokenizer', 'text_encoder', 'vae', 'unet'])


def create_model_weight_copies(model_dir: Path):
    for subfolder, (source_name, target_name) in MODEL_WEIGHT_ALIASES.items():
        source_path = model_dir / subfolder / source_name
        target_path = model_dir / subfolder / target_name
        if target_path.exists() or not source_path.is_file():
            continue
        shutil.copy2(source_path, target_path)


def prepare_sd_model():
    if has_diffusers_structure(PRETRAINED_MODEL_DIR) and not FORCE_MODEL_REDOWNLOAD:
        print('Modello SD2.1 gia\' presente.')
        create_model_weight_copies(PRETRAINED_MODEL_DIR)
        return
    if not PRETRAINED_MODEL_ZIP_PATH.exists():
        gdown.download(id=SD21_MODEL_DRIVE_ID, output=str(PRETRAINED_MODEL_ZIP_PATH), quiet=False)
    with TemporaryDirectory(prefix='sd21_extract_') as tmp:
        with zipfile.ZipFile(PRETRAINED_MODEL_ZIP_PATH) as archive:
            archive.extractall(tmp)
        for path in Path(tmp).rglob('model_index.json'):
            src_dir = path.parent
            if PRETRAINED_MODEL_DIR.exists():
                shutil.rmtree(PRETRAINED_MODEL_DIR)
            shutil.copytree(src_dir, PRETRAINED_MODEL_DIR)
            create_model_weight_copies(PRETRAINED_MODEL_DIR)
            break

prepare_sd_model()

# Opzionalmente sostituisce il VAE con la variante fine-tuned di 03. A differenza di
# un singolo path hardcoded, resolve_vae_ft_03_dir() prova piu' candidati (vedi Sezione 3)
# e solleva un errore chiaro se USE_VAE_FT_FROM_03=True ma nessuno e' disponibile,
# invece di ricadere silenziosamente sul VAE originale.
if USE_VAE_FT_FROM_03:
    vae_ft_dir = resolve_vae_ft_03_dir()
    if vae_ft_dir is None:
        candidates_str = '\n  - '.join(str(p) for p in vae_ft_03_candidate_dirs())
        raise FileNotFoundError(
            'USE_VAE_FT_FROM_03=True ma non ho trovato il VAE fine-tuned di 03 in nessuno '
            f'dei path candidati:\n  - {candidates_str}\n'
            'Esegui prima 03_SD21_VAE_FineTuned.ipynb oppure imposta '
            'USE_VAE_FT_FROM_03=False.'
        )
    dst = PRETRAINED_MODEL_DIR / 'vae'
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(vae_ft_dir, dst)
    create_model_weight_copies(PRETRAINED_MODEL_DIR)
    print('VAE fine-tuned di 03 copiato da:', vae_ft_dir, '->', dst)
    VAE_SOURCE = 'sd_vae_finetuned_03'
else:
    VAE_SOURCE = 'sd_vae_original'

SD_VAE_MODEL = PRETRAINED_MODEL_DIR
print('SD_VAE_MODEL:', SD_VAE_MODEL)
print('VAE_SOURCE  :', VAE_SOURCE)


Modello SD2.1 gia' presente.
SD_VAE_MODEL: /mnt/MammoDiffusion/MammoDiffusion/notebooks/pretrained_model/stable-diffusion-2-1-base
VAE_SOURCE  : sd_vae_original


## 6. Encoding latenti con SD-VAE (o VAE fine-tuned di 03)

Riusa lo stesso script di 07 (`prepare_sdvae_latents_v2.py`). Se `USE_VAE_FT_FROM_03=True`,
lo script legge il VAE che abbiamo appena copiato nella sotto-cartella `vae/` del modello base.

In [6]:
def run_and_stream(cmd, log_path, env=None):
    print(' '.join(str(c) for c in cmd))
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env or os.environ.copy())
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w', encoding='utf-8') as handle:
        for line in iter(process.stdout.readline, ''):
            handle.write(line)
            print(line, end='', flush=True)
    process.wait()
    if process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, cmd)


SDVAE_BATCH_SIZE = 4
FORCE_LATENTS_RECOMPUTE = False

prepare_cmd = [
    sys.executable,
    str(UTILITY_DIR / 'prepare_sdvae_latents_v2.py'),
    '--project-root', str(PROJECT_ROOT),
    '--experiment-dir', str(EXPERIMENT_DIR),
    '--batch-size', str(SDVAE_BATCH_SIZE),
    '--results-stage-name', RESULTS_STAGE_NAME,
    '--sd-vae-model', str(SD_VAE_MODEL),
]
if FORCE_LATENTS_RECOMPUTE:
    prepare_cmd.append('--force-recompute')

run_and_stream(prepare_cmd, LOGS_DIR / 'prepare_sdvae_latents.log')


/home/fede/miniforge3/envs/tf-gpu/bin/python /mnt/MammoDiffusion/MammoDiffusion/notebooks/utility/prepare_sdvae_latents_v2.py --project-root /mnt/MammoDiffusion/MammoDiffusion --experiment-dir /mnt/MammoDiffusion/MammoDiffusion/experiments/diffusers/08_ldm_v3_sdvae_fromscratch --batch-size 4 --results-stage-name diffusers/08_ldm_v3_sdvae_fromscratch --sd-vae-model /mnt/MammoDiffusion/MammoDiffusion/notebooks/pretrained_model/stable-diffusion-2-1-base
Uso metadata augmentation esistente: /mnt/MammoDiffusion/MammoDiffusion/data/real_augmented/metadata.csv
label
0    1701
1    1360
Name: count, dtype: int64
Latenti SD-VAE gia' presenti e coerenti: /mnt/MammoDiffusion/MammoDiffusion/experiments/diffusers/08_ldm_v3_sdvae_fromscratch/latents


## 7. Training U-Net v3 (v-prediction + min-SNR)

Lancia `train_ldm_v2.py` con i flag v3 definiti nella Sezione 3 (`UNET_VERSION`,
`PARAMETERIZATION`, `USE_MIN_SNR`, `MIN_SNR_GAMMA`), piu' `--vae-source` e
`--uses-vae-ft-from-03c` (nome legacy dell'argomento CLI) per tracciare nel manifest quale VAE e' stato usato:

* `--unet-version v3` -> il builder importa `ldm_v3_unet_keras.build_ldm_unet_v3`;
* `--parameterization v` -> loss simple = `||v_target - v_pred||^2`;
* `--use-min-snr --min-snr-gamma 5.0` -> weighting Min-SNR-gamma (formula corretta per v-prediction);
* `--vae-source` / `--uses-vae-ft-from-03c` -> salvati in `training_manifest.json`, letto
  automaticamente da generazione/valutazione/04c.

Al termine del training, `train_ldm_v2.py` scrive sempre `training_manifest.json` in
`EXPERIMENT_DIR`, cosi' generazione e valutazione a valle possono rilevare
la parameterization invece di doverla assumere.

Manteniamo gli stessi ordini di grandezza di 07 (150k step) per confrontabilita' diretta.


In [7]:
TOTAL_STEPS = 150_000
CHECKPOINT_EVERY = 5_000
LOG_EVERY = 20
RESUME_FROM_LATEST = True

train_cmd = [
    sys.executable,
    str(UTILITY_DIR / 'train_ldm_v2.py'),
    '--project-root', str(PROJECT_ROOT),
    '--experiment-dir', str(EXPERIMENT_DIR),
    '--total-steps', str(TOTAL_STEPS),
    '--checkpoint-every', str(CHECKPOINT_EVERY),
    '--log-every', str(LOG_EVERY),
    '--results-stage-name', RESULTS_STAGE_NAME,
    '--skip-latent-encoding',
    '--unet-version', UNET_VERSION,
    '--parameterization', PARAMETERIZATION,
    '--vae-source', VAE_SOURCE,
    '--notebook-name', NOTEBOOK_NAME,
]
if USE_MIN_SNR:
    train_cmd.append('--use-min-snr')
    train_cmd.extend(['--min-snr-gamma', str(MIN_SNR_GAMMA)])
if USE_VAE_FT_FROM_03:
    train_cmd.append('--uses-vae-ft-from-03c')
if RESUME_FROM_LATEST:
    train_cmd.append('--resume-from-latest')

run_and_stream(train_cmd, LOGS_DIR / 'ldm_train_v3.log', env=training_subprocess_env())


/home/fede/miniforge3/envs/tf-gpu/bin/python /mnt/MammoDiffusion/MammoDiffusion/notebooks/utility/train_ldm_v2.py --project-root /mnt/MammoDiffusion/MammoDiffusion --experiment-dir /mnt/MammoDiffusion/MammoDiffusion/experiments/diffusers/08_ldm_v3_sdvae_fromscratch --total-steps 150000 --checkpoint-every 5000 --log-every 20 --results-stage-name diffusers/08_ldm_v3_sdvae_fromscratch --skip-latent-encoding --unet-version v3 --parameterization v --vae-source sd_vae_original --notebook-name 08_LDM_v3_SDVAE_FromScratch.ipynb --use-min-snr --min-snr-gamma 5.0 --resume-from-latest
2026-07-11 17:18:45.407466: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-07-11 17:18:45.407494: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registere

## 8. Sweep dei checkpoint, generazione, filtro e valutazione finale

Le celle seguenti richiamano `evaluate_ldm_v2.py` e `generate_ldm_v2.py` con la stessa
logica di `07`, ma **non** riusano le celle di `07` per exec dinamico: sono celle
autonome che passano esplicitamente `--unet-version v3 --parameterization v` (e
`--vae-source` / `--uses-vae-ft-from-03c`), cosi' non c'e' rischio che una futura modifica
a `07` cambi silenziosamente il comportamento del notebook `08`, e non c'e' rischio che venga
dimenticata la conversione v-prediction -> epsilon durante generazione/valutazione.

Ogni comando include anche `--notebook-name` per tracciabilita' nei manifest/JSON prodotti.


### 8.1 Sweep checkpoint (FID/IS su validation)

In [8]:
EVAL_MIN_STEP = 1_000
N_GEN_PER_CLASS = 100
EVAL_SAMPLE_STEPS = 100
EVAL_GUIDANCE_SCALE = 1.5
EVAL_INCEPTION_BATCH = 8
EVAL_DECODE_ON_CPU = False
EVAL_ECO_TRACK = True

eval_cmd = [
    sys.executable,
    str(UTILITY_DIR / "evaluate_ldm_v2.py"),
    "--project-root", str(PROJECT_ROOT),
    "--experiment-dir", str(EXPERIMENT_DIR),
    "--mode", "both",
    "--min-step", str(EVAL_MIN_STEP),
    "--n-gen-per-class", str(N_GEN_PER_CLASS),
    "--sample-steps", str(EVAL_SAMPLE_STEPS),
    "--guidance-scale", str(EVAL_GUIDANCE_SCALE),
    "--mini-batch", "1",
    "--inception-batch", str(EVAL_INCEPTION_BATCH),
    "--results-stage-name", RESULTS_STAGE_NAME,
    "--vae-backend", "sd",
    "--sd-vae-model", str(SD_VAE_MODEL),
    "--unet-version", UNET_VERSION,
    "--parameterization", PARAMETERIZATION,
    "--vae-source", VAE_SOURCE,
    "--notebook-name", NOTEBOOK_NAME,
]
if USE_VAE_FT_FROM_03:
    eval_cmd.append("--uses-vae-ft-from-03c")
if EVAL_DECODE_ON_CPU:
    eval_cmd.append("--decode-on-cpu")
if EVAL_ECO_TRACK:
    eval_cmd.append("--eco-track")

add_generation_parallel_args(eval_cmd)
run_and_stream(eval_cmd, LOGS_DIR / "ldm_evaluate_sdvae_v3.log")


/home/fede/miniforge3/envs/tf-gpu/bin/python /mnt/MammoDiffusion/MammoDiffusion/notebooks/utility/evaluate_ldm_v2.py --project-root /mnt/MammoDiffusion/MammoDiffusion --experiment-dir /mnt/MammoDiffusion/MammoDiffusion/experiments/diffusers/08_ldm_v3_sdvae_fromscratch --mode both --min-step 1000 --n-gen-per-class 100 --sample-steps 100 --guidance-scale 1.5 --mini-batch 1 --inception-batch 8 --results-stage-name diffusers/08_ldm_v3_sdvae_fromscratch --vae-backend sd --sd-vae-model /mnt/MammoDiffusion/MammoDiffusion/notebooks/pretrained_model/stable-diffusion-2-1-base --unet-version v3 --parameterization v --vae-source sd_vae_original --notebook-name 08_LDM_v3_SDVAE_FromScratch.ipynb --eco-track --generation-gpus auto
Checkpoint candidati: 30
  step_5000 -> ldm_step005000.keras
  step_10000 -> ldm_step010000.keras
  step_15000 -> ldm_step015000.keras
  step_20000 -> ldm_step020000.keras
  step_25000 -> ldm_step025000.keras
  step_30000 -> ldm_step030000.keras
  step_35000 -> ldm_step0350

### 8.1b Ottimizzazione dei parametri di sampling sul best checkpoint

Sostituisce il precedente notebook standalone di ottimizzazione del sampler: qui
l'ottimizzazione viene fatta direttamente sopra il best checkpoint di questa
LDM v3 from-scratch (che era l'ultimo test di modello from-scratch previsto).

Idea: fissato il checkpoint scelto in 8.1, si stima *quanti* step di DDIM sono
davvero necessari per raggiungere la qualita' che la cella 8.1 ha misurato a
100 step. Un numero di step piu' basso a parita' di FID accelera la generazione
finale (8.2/8.3) e la valutazione, riducendo tempo e CO2.

Cosa fa la cella successiva:

1. Legge `evaluation/checkpoint_metrics.json` prodotto da 8.1, individua il
   `best_step` scelto (quello con `avg_FID` minimo).
2. Per ogni budget in `SAMPLING_STEP_BUDGETS` (default 25, 50, 75, 100), lancia
   `evaluate_ldm_v2.py` sul solo `step_{best_step}` con `--sample-steps N` e
   `--n-gen-per-class 50` (poche immagini bastano per una stima di FID
   affidabile in confronto relativo). L'output `checkpoint_metrics.json` viene
   spostato in `evaluation/sampling_sweep/steps_{N}_metrics.json` per non
   sovrascrivere quello di 8.1.
3. Aggrega i risultati in un DataFrame e salva
   `sampling_sweep_summary.csv`, senza generare plot dedicati.
4. Suggerisce `GEN_SAMPLE_STEPS_RECOMMENDED`: il minimo budget che raggiunge
   FID <= (FID a 100 step) * `SAMPLING_QUALITY_TOLERANCE` (default 1.05).
   Le celle 8.2/8.3 continuano a usare `GEN_SAMPLE_STEPS = 100` come default
   sicuro; e' la sezione 8.1b a dire se scendere e a quale valore.

La sezione e' **skippata automaticamente** se `checkpoint_metrics.json` non
esiste ancora (8.1 non e' stata eseguita) o se `RUN_SAMPLING_SWEEP = False`.


In [9]:
# Ottimizzazione sampler sul best checkpoint.
# evaluate_ldm_v2.py salva metriche nello schema v2: checkpoints + selection.

RUN_SAMPLING_SWEEP = True
SAMPLING_STEP_BUDGETS = [25, 50, 75, 100]
SAMPLING_N_GEN_PER_CLASS = 50   # 50 immagini/classe bastano per un confronto relativo affidabile
SAMPLING_QUALITY_TOLERANCE = 1.05  # accetta un budget se FID <= 1.05 * FID(100 step)

evaluation_dir = EXPERIMENT_DIR / "evaluation"
checkpoint_metrics_path = evaluation_dir / "checkpoint_metrics.json"
sampling_sweep_dir = evaluation_dir / "sampling_sweep"
sampling_sweep_dir.mkdir(parents=True, exist_ok=True)
sampling_sweep_csv = sampling_sweep_dir / "sampling_sweep_summary.csv"

GEN_SAMPLE_STEPS_RECOMMENDED = None

if not RUN_SAMPLING_SWEEP:
    print("Sampling sweep disattivato (RUN_SAMPLING_SWEEP=False): skip.")
elif not checkpoint_metrics_path.is_file():
    print(f"checkpoint_metrics.json non trovato in {checkpoint_metrics_path}. "
          "Esegui prima la cella 8.1, poi torna qui.")
else:
    import pandas as pd

    with checkpoint_metrics_path.open(encoding="utf-8") as handle:
        eval_payload = json.load(handle)

    if eval_payload.get("schema_version") != 2:
        raise RuntimeError("checkpoint_metrics.json non usa lo schema v2: riesegui la cella 8.1.")
    selection = eval_payload.get("selection", {})
    best_checkpoint_id = selection.get("best_checkpoint_id")
    selection_metric = selection.get("selection_metric")
    checkpoints = eval_payload.get("checkpoints", [])
    if not best_checkpoint_id or not selection_metric or not checkpoints:
        raise RuntimeError("checkpoint_metrics.json v2 incompleto: riesegui la cella 8.1.")
    try:
        best_row = next(row for row in checkpoints if row["checkpoint_id"] == best_checkpoint_id)
        fid_at_100 = float(best_row[selection_metric])
    except (KeyError, StopIteration, TypeError) as exc:
        raise RuntimeError("Metriche del best checkpoint mancanti: riesegui la cella 8.1.") from exc
    best_step = int(selection.get("best_step") or best_checkpoint_id.removeprefix("step_"))
    print(f"Best checkpoint da 8.1: {best_checkpoint_id} ({selection_metric} @ {EVAL_SAMPLE_STEPS} step = {fid_at_100:.3f})")

    # Il comando di sweep sovrascrive temporaneamente checkpoint_metrics.json.
    # La copia qui sotto viene ripristinata al termine, senza perdere la baseline di 8.1.
    baseline_metrics_backup = sampling_sweep_dir / "baseline_checkpoint_metrics.json"
    shutil.copy2(checkpoint_metrics_path, baseline_metrics_backup)
    records = []
    for steps in SAMPLING_STEP_BUDGETS:
        target_metrics_json = sampling_sweep_dir / f"steps_{steps}_checkpoint_metrics.json"
        if target_metrics_json.is_file():
            print(f"[{steps} step] gia' calcolato -> riuso {target_metrics_json.name}")
        else:
            print(f"[{steps} step] eseguo evaluate_ldm_v2.py --checkpoint-id {best_checkpoint_id} --sample-steps {steps}")
            sweep_eval_cmd = [
                sys.executable,
                str(UTILITY_DIR / "evaluate_ldm_v2.py"),
                "--project-root", str(PROJECT_ROOT),
                "--experiment-dir", str(EXPERIMENT_DIR),
                "--mode", "both",
                "--checkpoint-id", best_checkpoint_id,
                "--n-gen-per-class", str(SAMPLING_N_GEN_PER_CLASS),
                "--sample-steps", str(steps),
                "--guidance-scale", str(EVAL_GUIDANCE_SCALE),
                "--mini-batch", "1",
                "--inception-batch", str(EVAL_INCEPTION_BATCH),
                "--results-stage-name", f"{RESULTS_STAGE_NAME}_sampling_sweep_{steps}",
                "--vae-backend", "sd",
                "--sd-vae-model", str(SD_VAE_MODEL),
                "--unet-version", UNET_VERSION,
                "--parameterization", PARAMETERIZATION,
                "--vae-source", VAE_SOURCE,
                "--notebook-name", NOTEBOOK_NAME,
                "--force-recompute",
            ]
            if USE_VAE_FT_FROM_03:
                sweep_eval_cmd.append("--uses-vae-ft-from-03c")
            add_generation_parallel_args(sweep_eval_cmd)
            run_and_stream(sweep_eval_cmd, LOGS_DIR / f"sampling_sweep_{steps}steps.log")
            if not checkpoint_metrics_path.is_file():
                raise RuntimeError("evaluate_ldm_v2.py non ha prodotto checkpoint_metrics.json.")
            shutil.copy2(checkpoint_metrics_path, target_metrics_json)

        with target_metrics_json.open(encoding="utf-8") as handle:
            payload = json.load(handle)
        row = next(c for c in payload.get("checkpoints", []) if c.get("checkpoint_id") == best_checkpoint_id)
        wall_time = 0.0  # Lo schema v2 non salva un wall-time comparabile per checkpoint.
        records.append({
            "sample_steps": steps,
            "avg_FID": float(row[selection_metric]),
            "avg_IS_mean": float(row.get("is_mean_avg", float("nan"))),
            "avg_precision": float(row.get("precision_mean", float("nan"))),
            "avg_recall": float(row.get("recall_mean", float("nan"))),
            "seconds_per_image": float(wall_time) / max(1, 2 * SAMPLING_N_GEN_PER_CLASS) if wall_time else float("nan"),
        })

    # Ripristina la baseline e rigenera i suoi artefatti (CSV/plot/manifest).
    shutil.copy2(baseline_metrics_backup, checkpoint_metrics_path)
    restore_cmd = list(eval_cmd)
    restore_cmd[restore_cmd.index("--mode") + 1] = "artifacts"
    run_and_stream(restore_cmd, LOGS_DIR / "restore_baseline_evaluation_artifacts.log")

    df_sweep = pd.DataFrame(records).sort_values("sample_steps").reset_index(drop=True)
    df_sweep.to_csv(sampling_sweep_csv, index=False)
    print("\nRiepilogo sampling sweep:")
    print(df_sweep.to_string(index=False))

    print("Plot sampling sweep disattivati: metriche tabellari salvate in", sampling_sweep_csv)

    # Il riferimento deve avere lo stesso numero di immagini dello sweep (50/classe).
    # Se 100 non e' tra i budget, resta il valore baseline prodotto in 8.1.
    matching_reference = df_sweep[df_sweep["sample_steps"] == EVAL_SAMPLE_STEPS]
    if not matching_reference.empty:
        fid_at_100 = float(matching_reference["avg_FID"].iloc[0])

    # Raccomandazione: minimo budget con FID entro tolleranza rispetto al reference @ 100 step.
    tol_target = fid_at_100 * SAMPLING_QUALITY_TOLERANCE
    acceptable = df_sweep[df_sweep["avg_FID"] <= tol_target]
    if acceptable.empty:
        GEN_SAMPLE_STEPS_RECOMMENDED = int(df_sweep["sample_steps"].iloc[-1])
        print(f"\nNessun budget entro tolleranza {SAMPLING_QUALITY_TOLERANCE:.0%} vs FID @ {EVAL_SAMPLE_STEPS} step ({fid_at_100:.3f}): "
              f"resta consigliato il massimo ({GEN_SAMPLE_STEPS_RECOMMENDED} step).")
    else:
        GEN_SAMPLE_STEPS_RECOMMENDED = int(acceptable["sample_steps"].min())
        best_row_sweep = acceptable.iloc[0]
        print(f"\nRACCOMANDAZIONE: usa {GEN_SAMPLE_STEPS_RECOMMENDED} step "
              f"(FID {float(best_row_sweep['avg_FID']):.3f} vs riferimento {fid_at_100:.3f}, "
              f"entro {SAMPLING_QUALITY_TOLERANCE:.0%}). "
              f"Modifica GEN_SAMPLE_STEPS nelle celle 8.2/8.3 se accetti la raccomandazione.")

    with (sampling_sweep_dir / "recommendation.json").open("w", encoding="utf-8") as handle:
        json.dump({
            "best_checkpoint_id": best_checkpoint_id,
            "best_step": int(best_step),
            "reference_sample_steps": int(EVAL_SAMPLE_STEPS),
            "reference_fid": float(fid_at_100),
            "quality_tolerance": float(SAMPLING_QUALITY_TOLERANCE),
            "step_budgets_tested": list(SAMPLING_STEP_BUDGETS),
            "recommended_gen_sample_steps": (int(GEN_SAMPLE_STEPS_RECOMMENDED)
                                             if GEN_SAMPLE_STEPS_RECOMMENDED is not None else None),
            "sweep_summary_csv": str(sampling_sweep_csv),
        }, handle, indent=2, ensure_ascii=False)


Best checkpoint da 8.1: step_75000 (fid_1 @ 100 step = 114.498)
[25 step] gia' calcolato -> riuso steps_25_checkpoint_metrics.json
[50 step] gia' calcolato -> riuso steps_50_checkpoint_metrics.json
[75 step] gia' calcolato -> riuso steps_75_checkpoint_metrics.json
[100 step] gia' calcolato -> riuso steps_100_checkpoint_metrics.json
/home/fede/miniforge3/envs/tf-gpu/bin/python /mnt/MammoDiffusion/MammoDiffusion/notebooks/utility/evaluate_ldm_v2.py --project-root /mnt/MammoDiffusion/MammoDiffusion --experiment-dir /mnt/MammoDiffusion/MammoDiffusion/experiments/diffusers/08_ldm_v3_sdvae_fromscratch --mode artifacts --min-step 1000 --n-gen-per-class 100 --sample-steps 100 --guidance-scale 1.5 --mini-batch 1 --inception-batch 8 --results-stage-name diffusers/08_ldm_v3_sdvae_fromscratch --vae-backend sd --sd-vae-model /mnt/MammoDiffusion/MammoDiffusion/notebooks/pretrained_model/stable-diffusion-2-1-base --unet-version v3 --parameterization v --vae-source sd_vae_original --notebook-name 08_L

### 8.1c Smoke test multi-GPU isolato (opzionale)

Cella disattivata di default (`RUN_MULTI_GPU_SMOKE = False`). Se attivata, esegue una generazione RAW minima (`SMOKE_N_RAW` immagini, `SMOKE_SAMPLE_STEPS` step) in una directory dedicata e univoca sotto `EXPERIMENT_DIR/smoke_multi_gpu_dynamic/run_<time_ns>/`, per verificare che la generazione multi-GPU usi realmente almeno 2 worker/GPU prima di lanciare la generazione reale della sezione 8.2. Non tocca le directory RAW/FILTERED canoniche e non avvia filtro, validazione, test o reverse diffusion.


In [10]:
RUN_MULTI_GPU_SMOKE = False
SMOKE_GENERATION_GPU_DEVICES = "0,1"
SMOKE_MAX_WORKERS = 2
SMOKE_N_RAW = 16
SMOKE_SAMPLE_STEPS = 2
SMOKE_GENERATION_SCHEDULER = "dynamic_reservations"
SMOKE_RESERVATION_SIZE = 4
SMOKE_GUIDANCE_SCALE = 1.5
SMOKE_TARGET_LABEL = 1
SMOKE_PARAMETERIZATION = PARAMETERIZATION
SMOKE_UNET_VERSION = UNET_VERSION
SMOKE_VAE_SOURCE = VAE_SOURCE

if not RUN_MULTI_GPU_SMOKE:
    print("RUN_MULTI_GPU_SMOKE=False: smoke multi-GPU non eseguito (default).")
else:
    if str(UTILITY_DIR) not in sys.path:
        sys.path.insert(0, str(UTILITY_DIR))
    from generate_ldm_v2 import is_readable_png, resolve_model_path
    from parallel_generation_utils import print_gpu_resolution_dry_run

    print("CUDA_VISIBLE_DEVICES ereditato:", os.environ.get("CUDA_VISIBLE_DEVICES"))
    print("GENERATION_GPU_DEVICES richiesto (smoke):", SMOKE_GENERATION_GPU_DEVICES)
    # Lista esplicita, non "auto": lo smoke fissa le GPU indipendentemente da
    # GENERATION_GPU_DEVICES/CUDA_VISIBLE_DEVICES usati dalla generazione reale.
    smoke_devices = print_gpu_resolution_dry_run(SMOKE_GENERATION_GPU_DEVICES, SMOKE_MAX_WORKERS)
    if len(smoke_devices) < 2:
        raise RuntimeError(
            f"SMOKE MULTI-GPU FALLITO: risolte {len(smoke_devices)} GPU {smoke_devices}, "
            "ne servono almeno 2. Non continuo silenziosamente con una sola GPU."
        )

    SMOKE_MODEL_PATH = resolve_model_path(EXPERIMENT_DIR)
    SMOKE_RUN_DIR = EXPERIMENT_DIR / "smoke_multi_gpu_dynamic" / f"run_{time.time_ns()}"
    SMOKE_RAW_DIR = SMOKE_RUN_DIR / "raw"
    SMOKE_FILTERED_DIR = SMOKE_RUN_DIR / "filtered"
    SMOKE_LOGS_DIR = SMOKE_RUN_DIR / "logs"
    SMOKE_LOGS_DIR.mkdir(parents=True, exist_ok=True)

    # I worker multi-GPU della generazione RAW scrivono sempre sotto
    # EXPERIMENT_DIR/logs/parallel_generation/run_<ts> (create_parallel_run_dir),
    # non sotto SMOKE_RUN_DIR: la troviamo confrontando lo stato prima/dopo.
    parallel_logs_root = Path(EXPERIMENT_DIR).resolve() / "logs" / "parallel_generation"
    runs_before = set(parallel_logs_root.glob("run_*")) if parallel_logs_root.is_dir() else set()

    smoke_cmd = [
        sys.executable,
        str(UTILITY_DIR / "generate_ldm_v2.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--model-path", str(SMOKE_MODEL_PATH),
        "--mode", "generate",
        "--n-raw", str(SMOKE_N_RAW),
        "--target-label", str(SMOKE_TARGET_LABEL),
        "--raw-dir", str(SMOKE_RAW_DIR),
        "--filtered-dir", str(SMOKE_FILTERED_DIR),
        "--sample-steps", str(SMOKE_SAMPLE_STEPS),
        "--guidance-scale", str(SMOKE_GUIDANCE_SCALE),
        "--vae-backend", "sd",
        "--sd-vae-model", str(SD_VAE_MODEL),
        "--sd-vae-batch-size", "1",
        "--parameterization", SMOKE_PARAMETERIZATION,
        "--unet-version", SMOKE_UNET_VERSION,
        "--vae-source", SMOKE_VAE_SOURCE,
        "--generation-gpus", SMOKE_GENERATION_GPU_DEVICES,
        "--generation-scheduler", SMOKE_GENERATION_SCHEDULER,
        "--generation-reservation-size", str(SMOKE_RESERVATION_SIZE),
        "--max-generation-workers", str(SMOKE_MAX_WORKERS),
    ]
    run_and_stream(smoke_cmd, SMOKE_LOGS_DIR / "smoke_multi_gpu_generate.log")

    expected_smoke_pngs = [SMOKE_RAW_DIR / f"synth_{index:05d}.png" for index in range(SMOKE_N_RAW)]
    actual_smoke_pngs = sorted(SMOKE_RAW_DIR.glob("*.png"))
    if [p.name for p in actual_smoke_pngs] != [p.name for p in expected_smoke_pngs]:
        raise RuntimeError(
            f"SMOKE MULTI-GPU FALLITO: attesi {[p.name for p in expected_smoke_pngs]}, "
            f"trovati {[p.name for p in actual_smoke_pngs]} in {SMOKE_RAW_DIR}"
        )
    unreadable_smoke_pngs = [p.name for p in expected_smoke_pngs if not is_readable_png(p)]
    if unreadable_smoke_pngs:
        raise RuntimeError(f"SMOKE MULTI-GPU FALLITO: PNG non leggibili: {unreadable_smoke_pngs}")

    runs_after = set(parallel_logs_root.glob("run_*"))
    new_runs = sorted(runs_after - runs_before)
    if len(new_runs) != 1:
        raise RuntimeError(
            "SMOKE MULTI-GPU FALLITO: attesa esattamente 1 nuova directory log di run, "
            f"trovate {len(new_runs)}: {new_runs}"
        )
    smoke_run_log_dir = new_runs[0]
    smoke_worker_logs = sorted(smoke_run_log_dir.glob("*_gpu_*.log"))
    if len(smoke_worker_logs) < 2:
        raise RuntimeError(
            f"SMOKE MULTI-GPU FALLITO: attesi almeno 2 log worker distinti in {smoke_run_log_dir}, "
            f"trovati {len(smoke_worker_logs)}: {[p.name for p in smoke_worker_logs]}. "
            "Meno di due GPU/worker sono state effettivamente usate."
        )
    for worker_log in smoke_worker_logs:
        shutil.copy2(worker_log, SMOKE_LOGS_DIR / worker_log.name)

    print("Directory smoke:", SMOKE_RUN_DIR)
    print("Directory log worker:", smoke_run_log_dir)
    print("Log worker trovati:", [p.name for p in smoke_worker_logs])
    print("SMOKE MULTI-GPU SUPERATO")


RUN_MULTI_GPU_SMOKE=False: smoke multi-GPU non eseguito (default).


### 8.2 Generazione, filtro e test - classe positiva

In [11]:
GEN_N_RAW = 4083
GEN_N_SELECTED = 1361
GEN_SAMPLE_STEPS = 100
GEN_GUIDANCE_SCALE = 1.5
GEN_MODEL_PATH = CHECKPOINTS_DIR / "ldm_unet_best_eval.keras"
GEN_DECODE_ON_CPU = False
GEN_ECO_TRACK = True

pos_cmd = [
    sys.executable,
    str(UTILITY_DIR / "generate_ldm_v2.py"),
    "--project-root", str(PROJECT_ROOT),
    "--experiment-dir", str(EXPERIMENT_DIR),
    "--model-path", str(GEN_MODEL_PATH),
    "--mode", "all",
    "--n-raw", str(GEN_N_RAW),
    "--n-selected", str(GEN_N_SELECTED),
    "--target-label", "1",
    "--batch-size", "1",
    "--sample-steps", str(GEN_SAMPLE_STEPS),
    "--guidance-scale", str(GEN_GUIDANCE_SCALE),
    "--results-stage-name", RESULTS_STAGE_NAME,
    "--vae-backend", "sd",
    "--sd-vae-model", str(SD_VAE_MODEL),
    "--unet-version", UNET_VERSION,
    "--parameterization", PARAMETERIZATION,
    "--vae-source", VAE_SOURCE,
    "--notebook-name", NOTEBOOK_NAME,
]
if USE_VAE_FT_FROM_03:
    pos_cmd.append("--uses-vae-ft-from-03c")
if GEN_DECODE_ON_CPU:
    pos_cmd.append("--decode-on-cpu")
if GEN_ECO_TRACK:
    pos_cmd.append("--eco-track")

# "--mode all" scrive nei path canonici dell'esperimento (get_experiment_paths), che per
# EXPERIMENT_NAME=diffusers/08_ldm_v3_sdvae_fromscratch risolvono a SYNTHETIC_V3_POS_DIR
# (vedi ldm_project_paths.FILTERED_DIR_NAME_BY_EXPERIMENT) -- niente collisione con
# 07.
add_generation_parallel_args(pos_cmd)
run_and_stream(pos_cmd, LOGS_DIR / "ldm_generate_positive_sdvae_v3.log")


/home/fede/miniforge3/envs/tf-gpu/bin/python /mnt/MammoDiffusion/MammoDiffusion/notebooks/utility/generate_ldm_v2.py --project-root /mnt/MammoDiffusion/MammoDiffusion --experiment-dir /mnt/MammoDiffusion/MammoDiffusion/experiments/diffusers/08_ldm_v3_sdvae_fromscratch --model-path /mnt/MammoDiffusion/MammoDiffusion/experiments/diffusers/08_ldm_v3_sdvae_fromscratch/checkpoints_ldm/ldm_unet_best_eval.keras --mode all --n-raw 4083 --n-selected 1361 --target-label 1 --batch-size 1 --sample-steps 100 --guidance-scale 1.5 --results-stage-name diffusers/08_ldm_v3_sdvae_fromscratch --vae-backend sd --sd-vae-model /mnt/MammoDiffusion/MammoDiffusion/notebooks/pretrained_model/stable-diffusion-2-1-base --unet-version v3 --parameterization v --vae-source sd_vae_original --notebook-name 08_LDM_v3_SDVAE_FromScratch.ipynb --eco-track --generation-gpus auto


Orchestrazione in subprocess separati: generate, filter, validate, test

[orchestrator] /home/fede/miniforge3/envs/tf-gpu/bin/python /mnt/MammoDiffusion/MammoDiffusion/notebooks/utility/generate_ldm_v2.py --mode generate --project-root /mnt/MammoDiffusion/MammoDiffusion --experiment-dir /mnt/MammoDiffusion/MammoDiffusion/experiments/diffusers/08_ldm_v3_sdvae_fromscratch --cuda-root /home/fede/miniforge3/envs/tf-gpu --target-label 1 --n-raw 4083 --n-selected 1361 --batch-size 1 --sample-steps 100 --guidance-scale 1.5 --nonblack-threshold 10 --vram-log-every 25 --seed 42 --balanced-seed 42 --inception-batch 8 --is-splits 10 --knn-k 3 --inception-weights imagenet --results-stage-name diffusers/08_ldm_v3_sdvae_fromscratch --generation-gpus auto --model-path /mnt/MammoDiffusion/MammoDiffusion/experiments/diffusers/08_ldm_v3_sdvae_fromscratch/checkpoints_ldm/ldm_unet_best_eval.keras --vae-backend sd --sd-vae-model /mnt/MammoDiffusion/MammoDiffusion/notebooks/pretrained_model/stable-diffusion

### 8.3 Generazione, filtro, validazione e test - classe negativa


In [12]:
NEG_RAW_DIR = EXPERIMENT_DIR / "synthetic_raw_negative"
NEG_FILTERED_DIR = SYNTHETIC_V3_NEG_DIR

neg_base_cmd = [
    sys.executable,
    str(UTILITY_DIR / "generate_ldm_v2.py"),
    "--project-root", str(PROJECT_ROOT),
    "--experiment-dir", str(EXPERIMENT_DIR),
    "--model-path", str(GEN_MODEL_PATH),
    "--n-raw", str(GEN_N_RAW),
    "--n-selected", str(GEN_N_SELECTED),
    "--target-label", "0",
    "--raw-dir", str(NEG_RAW_DIR),
    "--filtered-dir", str(NEG_FILTERED_DIR),
    "--batch-size", "1",
    "--sample-steps", str(GEN_SAMPLE_STEPS),
    "--guidance-scale", str(GEN_GUIDANCE_SCALE),
    "--results-stage-name", RESULTS_STAGE_NAME,
    "--vae-backend", "sd",
    "--sd-vae-model", str(SD_VAE_MODEL),
    "--unet-version", UNET_VERSION,
    "--parameterization", PARAMETERIZATION,
    "--vae-source", VAE_SOURCE,
    "--notebook-name", NOTEBOOK_NAME,
]
if USE_VAE_FT_FROM_03:
    neg_base_cmd.append("--uses-vae-ft-from-03c")
if GEN_DECODE_ON_CPU:
    neg_base_cmd.append("--decode-on-cpu")
if GEN_ECO_TRACK:
    neg_base_cmd.append("--eco-track")

add_generation_parallel_args(neg_base_cmd)
neg_cmd = [*neg_base_cmd, "--mode", "all"]
run_and_stream(neg_cmd, LOGS_DIR / "ldm_negative_sdvae_v3_all.log")


/home/fede/miniforge3/envs/tf-gpu/bin/python /mnt/MammoDiffusion/MammoDiffusion/notebooks/utility/generate_ldm_v2.py --project-root /mnt/MammoDiffusion/MammoDiffusion --experiment-dir /mnt/MammoDiffusion/MammoDiffusion/experiments/diffusers/08_ldm_v3_sdvae_fromscratch --model-path /mnt/MammoDiffusion/MammoDiffusion/experiments/diffusers/08_ldm_v3_sdvae_fromscratch/checkpoints_ldm/ldm_unet_best_eval.keras --n-raw 4083 --n-selected 1361 --target-label 0 --raw-dir /mnt/MammoDiffusion/MammoDiffusion/experiments/diffusers/08_ldm_v3_sdvae_fromscratch/synthetic_raw_negative --filtered-dir /mnt/MammoDiffusion/MammoDiffusion/data/synthetic/fromscratch_v3/negative --batch-size 1 --sample-steps 100 --guidance-scale 1.5 --results-stage-name diffusers/08_ldm_v3_sdvae_fromscratch --vae-backend sd --sd-vae-model /mnt/MammoDiffusion/MammoDiffusion/notebooks/pretrained_model/stable-diffusion-2-1-base --unet-version v3 --parameterization v --vae-source sd_vae_original --notebook-name 08_LDM_v3_SDVAE_Fro

### 8.4 Riepilogo artefatti


In [13]:
import json as _json
from pathlib import Path
import pandas as pd

summary = {
    "best_eval": CHECKPOINTS_DIR / "ldm_unet_best_eval.keras",
    "training_manifest": EXPERIMENT_DIR / "training_manifest.json",
    "latent_stats": LATENTS_DIR / "latent_stats.npz",
    "positive_final_json": RESULTS_METRICS_DIR / "positive" / "final_filtered_vs_test.json",
    "negative_final_json": RESULTS_METRICS_DIR / "negative" / "final_filtered_vs_test.json",
    "positive_filtered_dir": SYNTHETIC_V3_POS_DIR,
    "negative_filtered_dir": SYNTHETIC_V3_NEG_DIR,
}
for name, path in summary.items():
    print(f"{name:22s}", path, "OK" if Path(path).exists() else "MISSING")

if summary["training_manifest"].exists():
    with open(summary["training_manifest"], encoding="utf-8") as handle:
        training_manifest = _json.load(handle)
    print("\ntraining_manifest.json:")
    print(_json.dumps(training_manifest, indent=2, ensure_ascii=False))

rows = []
for label_name, directory in [("positive", SYNTHETIC_V3_POS_DIR), ("negative", SYNTHETIC_V3_NEG_DIR)]:
    rows.append({
        "class": label_name,
        "directory": str(directory),
        "n_png": len(sorted(Path(directory).glob("*.png"))) if Path(directory).is_dir() else 0,
    })
pd.DataFrame(rows)


best_eval              /mnt/MammoDiffusion/MammoDiffusion/experiments/diffusers/08_ldm_v3_sdvae_fromscratch/checkpoints_ldm/ldm_unet_best_eval.keras OK
training_manifest      /mnt/MammoDiffusion/MammoDiffusion/experiments/diffusers/08_ldm_v3_sdvae_fromscratch/training_manifest.json OK
latent_stats           /mnt/MammoDiffusion/MammoDiffusion/experiments/diffusers/08_ldm_v3_sdvae_fromscratch/latents/latent_stats.npz OK
positive_final_json    /mnt/MammoDiffusion/MammoDiffusion/results/diffusers/08_ldm_v3_sdvae_fromscratch/metrics/positive/final_filtered_vs_test.json OK
negative_final_json    /mnt/MammoDiffusion/MammoDiffusion/results/diffusers/08_ldm_v3_sdvae_fromscratch/metrics/negative/final_filtered_vs_test.json OK
positive_filtered_dir  /mnt/MammoDiffusion/MammoDiffusion/data/synthetic/fromscratch_v3/positive OK
negative_filtered_dir  /mnt/MammoDiffusion/MammoDiffusion/data/synthetic/fromscratch_v3/negative OK

training_manifest.json:
{
  "notebook": "08_LDM_v3_SDVAE_FromScratch.ipyn

,class,directory,n_png
0,positive,/mnt/MammoDiffusion/MammoDiffusion/data/synthe...,1361
1,negative,/mnt/MammoDiffusion/MammoDiffusion/data/synthe...,1361
